# Build the `Workspace_Access` Direct Lake model

Creates a Direct Lake semantic model over the workspace-access tables and adds measures
for the *"where do individual users have access"* report — how much access is granted
directly to users vs Entra groups / service principals, and who holds write-level access.

Reads two tables:
- **`workspace_access`** — every (workspace, principal) assignment (users, groups, SPs)
- **`workspace_user_access_summary`** — per-user rollup for the drill-down table

## Snapshot note
Both are **daily snapshots** (partitioned by `snapshot_date`). The measures below pin to
the **latest** snapshot so repeated runs don't double-count; add a `snapshot_date` slicer
to trend over time, or to filter the raw-column table visuals.

## Prerequisites
- `workspace_access_to_lakehouse` has run, so the tables exist.
- Workspace **XMLA endpoint = Read Write** (for the measure step).

In [ ]:
%pip install -q semantic-link-labs "PyJWT>=2.6.0"

In [ ]:
from sempy_labs.directlake import generate_direct_lake_semantic_model

MODEL  = "Workspace_Access"
LH     = "lh_fabric_management"
SCHEMA = "fabricmanagement"
TABLES = {
    "workspace_access":              f"{SCHEMA}.workspace_access",
    "workspace_user_access_summary": f"{SCHEMA}.workspace_user_access_summary",
}

generate_direct_lake_semantic_model(
    dataset=MODEL, tables=TABLES, source=LH, source_type="Lakehouse",
    overwrite=True, refresh=True,
)
print(f"Direct Lake model '{MODEL}' created over {list(TABLES)}")

In [ ]:
from sempy_labs.tom import connect_semantic_model

T = "workspace_access"

# Filter fragments (single-column predicates -> valid CALCULATE filters)
USER  = "'workspace_access'[principal_type] = \"User\""
GROUP = "'workspace_access'[principal_type] = \"Group\""
SP    = "'workspace_access'[principal_type] = \"ServicePrincipal\""
WRITE = "'workspace_access'[is_write_role] = TRUE()"
ADMIN = "'workspace_access'[role] = \"Admin\""

def latest(expr, *extra):
    # Pin the measure to the most recent snapshot_date, then apply any extra filters.
    filters = ["'workspace_access'[snapshot_date] = _last", *extra]
    return ("VAR _last = CALCULATE(MAX('workspace_access'[snapshot_date]), "
            "ALL('workspace_access')) "
            f"RETURN CALCULATE({expr}, {', '.join(filters)})")

with connect_semantic_model(dataset=MODEL, readonly=False) as tom:
    tom.add_measure(T, "Access Assignments",
                    latest("COUNTROWS('workspace_access')"), format_string="#,0")
    tom.add_measure(T, "Workspaces",
                    latest("DISTINCTCOUNT('workspace_access'[workspace_id])"), format_string="#,0")
    tom.add_measure(T, "User Assignments",
                    latest("COUNTROWS('workspace_access')", USER), format_string="#,0")
    tom.add_measure(T, "Group Assignments",
                    latest("COUNTROWS('workspace_access')", GROUP), format_string="#,0")
    tom.add_measure(T, "Service Principal Assignments",
                    latest("COUNTROWS('workspace_access')", SP), format_string="#,0")
    tom.add_measure(T, "Distinct Users",
                    latest("DISTINCTCOUNT('workspace_access'[principal_id])", USER), format_string="#,0")
    tom.add_measure(T, "Write-level User Assignments",
                    latest("COUNTROWS('workspace_access')", USER, WRITE), format_string="#,0")
    tom.add_measure(T, "Admin User Assignments",
                    latest("COUNTROWS('workspace_access')", USER, ADMIN), format_string="#,0")
    tom.add_measure(T, "% Access via Individual Users",
                    "DIVIDE([User Assignments], [Access Assignments])", format_string="0.0%")
print("Measures added.")

## Verify — measures

In [ ]:
from sempy_labs.tom import connect_semantic_model
print("Measures on Workspace_Access:")
with connect_semantic_model(dataset="Workspace_Access", readonly=True) as tom:
    for m in tom.all_measures():
        print(f"  [{m.Parent.Name}]  {m.Name}")

## Build the report (in the Fabric UI)

On the **`Workspace_Access`** model → **Auto-create report** (or New report):

1. **Cards:** `[User Assignments]`, `[% Access via Individual Users]`, `[Distinct Users]`,
   `[Write-level User Assignments]`.
2. **Bar — access by principal type (governance health):** `[User Assignments]`,
   `[Group Assignments]`, `[Service Principal Assignments]` — a high **% via Individual
   Users** is the anti-pattern to drive down.
3. **Table — per-user drill (where each user has access):** from
   `workspace_user_access_summary` → `email`, `user_name`, `n_workspaces`,
   `has_write_access`, `workspaces`; sort by `n_workspaces` desc. Add a `snapshot_date`
   slicer set to the latest date (this is a raw-column table, not measure-driven).
4. **Table — write-level detail:** from `workspace_access` → `workspace_name`,
   `display_name`, `email`, `role`; filter `principal_type = User` and
   `is_write_role = True`.

**Save as `Workspace Access & Direct-User Governance`.**

The users at the top of the per-user table — especially those with `has_write_access` —
are your candidates to move behind an Entra group or service principal.